# GDSN-DA Full-Mamba — BCI Competition IV-2a LOSO
Enable **GPU** and **Internet** in Kaggle, then run the cell below.

Source subjects: session T trains, session E validates (checkpoint selection). Target subject: session T is unlabeled adaptation data, session E is IM-TTA + primary test.

Set `STAGE` at the top of the cell: **1** = group-wise DSBN with a fixed adversary, **2** = + adaptive adversary (main method), **3** = + low-rank cross-group mixer. Use `SEEDS = [0, 1, 2]` for a multi-seed estimate.

In [ ]:
# One-cell Kaggle job: GDSN-DA Full-Mamba, BCI Competition IV-2a LOSO.
# Enable GPU + Internet in Kaggle, then run this cell.
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

# ---------------- Experiment switches ----------------
# STAGE 1: group-wise DSBN, fixed-strength adversary, no cross-group mixer
# STAGE 2: + discriminator-accuracy adaptive adversary   (main method)
# STAGE 3: + zero-gated low-rank cross-group mixer (rank 4)
STAGE = 2
SEEDS = [0]                      # e.g. [0, 1, 2] for a 3-seed estimate
TARGET_SUBJECTS = list(range(1, 10))
USE_IM_TTA = True
IM_TTA_STEPS = 5

STAGES = {
    1: dict(feature_alignment='group_dsbn', adaptive_adversary=False, cross_group_rank=0),
    2: dict(feature_alignment='group_dsbn', adaptive_adversary=True, cross_group_rank=0),
    3: dict(feature_alignment='group_dsbn', adaptive_adversary=True, cross_group_rank=4),
}
STAGE_KWARGS = STAGES[STAGE]

BRANCH = 'feature/hada-full-mamba-gdsn-da'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
DATASET = 'bcic2a'
RUN_TAG = f"{DATASET}_stage{STAGE}_imtta-{IM_TTA_STEPS if USE_IM_TTA else 0}_seeds-{'-'.join(map(str, SEEDS))}"
REPO_PATH = Path(f'/kaggle/working/tcformer-gdsn-da-{DATASET}')
MNE_DATA = Path('/kaggle/working/mne_data')
TRAIN_LOG = Path(f'/kaggle/working/gdsn_da_{RUN_TAG}.log')
RESULT_ARCHIVE = Path(f'/kaggle/working/gdsn_da_{RUN_TAG}_results')


def run(command, cwd=None, env=None, stream=False, log_path=None, append=False):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'a' if append else 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                heartbeat = f'[Kaggle heartbeat] running | elapsed={(time.monotonic() - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')


# ---------------- Clone and verify the branch ----------------
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    # A previous run edits the YAML config; force the disposable clone back.
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

model_text = (REPO_PATH / 'models/hada_tcformer.py').read_text(encoding='utf-8')
config_text = (REPO_PATH / 'configs/hada_tcformer.yaml').read_text(encoding='utf-8')
pipeline_text = (REPO_PATH / 'train_pipeline.py').read_text(encoding='utf-8')
assert 'class DomainSpecificNorm' in model_text
assert 'class LowRankCrossGroupMixer' in model_text
assert 'def _adversary_strength' in model_text
assert 'temporal_mmd_loss' not in model_text
assert 'sequence_block_types: [mamba_forward, mamba_backward, mamba_forward, mamba_backward, mamba_forward]' in config_text
assert 'model.load_state_dict(metrics_callback.best_model_state)' in pipeline_text
print(f'Verified GDSN-DA Full-Mamba branch | STAGE={STAGE} | {STAGE_KWARGS}')

# ---------------- Environment ----------------
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

# ---------------- Config override (stage + subjects) ----------------
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    f"c['model_kwargs'].update({STAGE_KWARGS!r}); "
    f"c['model_kwargs']['im_tta_steps']={IM_TTA_STEPS if USE_IM_TTA else 0}; "
    "c['model_kwargs']['log_every_n_batches']=0; "
    ""
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; print('GPU:', torch.cuda.get_device_name(0))"])

# ---------------- Train / test every seed ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
results_dir = REPO_PATH / 'results'
if results_dir.exists():
    # A rerun in the same draft session must not mix old and new results.
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    'PYTHONUNBUFFERED': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'MPLBACKEND': 'Agg',
    'MNE_DATA': str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
})

print(f'===== GDSN-DA FULL-MAMBA | BCI IV-2a | 9 SUBJECTS | 4 CLASSES | 22 CH | STAGE {STAGE} =====', flush=True)
print('Protocol: Source subjects: session T trains, session E validates (checkpoint selection). Target subject: session T is unlabeled adaptation data, session E is IM-TTA + primary test.', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
for index, seed in enumerate(SEEDS):
    print(f'\n===== SEED {seed} ({index + 1}/{len(SEEDS)}) =====', flush=True)
    run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', DATASET,
         '--loso', '--gpu_id', '0', '--seed', seed],
        cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG, append=index > 0)

# ---------------- Collect results ----------------
for summary in sorted(results_dir.rglob('*.txt')):
    text = summary.read_text(encoding='utf-8', errors='replace')
    if 'Average' in text:
        print(f'\n----- {summary.relative_to(REPO_PATH)} -----\n{text}')
shutil.copy(TRAIN_LOG, results_dir / TRAIN_LOG.name)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Kaggle output archive:', archive, flush=True)
